# Nemotron RFT — Generate model's own correct CoTs (vLLM, 100% offline)

**Goal:** lift LB from 0.86 → ~0.88–0.90 by training on the model's own
correct reasoning instead of the noisy original CoT.

**Engine:** vLLM with continuous batching + LoRARequest. Same pattern as the
official `nemotron-baseline-evaluation.ipynb` and `tinker-submission-notebook.ipynb`.

**Source of vLLM (offline):** the **NVIDIA metric utility script** at
`/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script` ←
**underscores** path. Auto-attached when you enable the competition metric.

## Round 1 results (already done — `rft_train.jsonl`, 703 examples)

| Category | Accept rate | Kept |
|---|---|---|
| equation_numeric_deduce | **26.2%** ✓ | 567 |
| equation_numeric_guess | 7.9% | 35 |
| bit_manipulation | 1.1% (truncated) | 60 |
| cryptarithm_deduce | 1.2% (truncated) | 32 |
| cryptarithm_guess | 1.4% (truncated) | 9 |

## Round 2 — TIGHT 12-hr budget

The first round 2 attempt (K=6, max_tok=14336, 2298 prompts) ran past the
12-hr Kaggle session and saved nothing because the single `llm.generate()`
call buffered all outputs in memory. This config is sized to **finish in
6-8 hours** AND **save progress every 200 prompts** so a timeout only
loses the in-flight chunk.

| Setting | Round 1 | First R2 attempt | Round 2 (this) | Why |
|---|---|---|---|---|
| `MAX_NEW_TOK` | 6144 | 14336 | **10240** | 67% more than R1, 28% less than R2-first |
| `max_model_len` | 8192 | 16384 | **12288** | Smaller KV cache → more concurrent seqs |
| `max_num_seqs` | 64 | 32 | **48** | Use the freed KV cache for concurrency |
| `K` (samples/prompt) | 4 | 6 | **4** | 33% less work vs R2-first |
| `TEMPERATURE` | 1.0 | 0.5 | **0.5** | Focused reasoning |
| `TOP_P` | 0.95 | 0.9 | **0.9** | Stricter nucleus |
| Categories | 5 hard | 4 hard | **4 hard** | Skip equation_numeric_deduce (done) |
| `MAX_PROMPTS_PER_CATEGORY` | unlim | unlim | **400** | Caps bit_manip 1364→400 |
| **Periodic save** | ❌ all-at-end | ❌ all-at-end | **✅ every 200** | Timeout safety |

**Workload:** ~1075 prompts × K=4 = 4300 completions, expected **5-8 hrs**.

**Expected yield:** 200-400 new examples → combined with round 1's 703 →
**~900-1100 total RFT training examples**.

## Required Kaggle inputs

- `metric/nemotron-3-nano-30b-a3b-bf16` (model)
- The **NVIDIA Nemotron metric utility script** (auto-attached when you enable
  the competition metric)
- **Your 0.86 adapter dataset** — set `ADAPTER_PATH` in cell 4
- Your training data dataset (cell 4 `DATA_DIR_CANDIDATES`)
- **Round 1 RFT data** (`rft_train.jsonl`) — for cell 9 to combine. Either
  upload as a Kaggle dataset OR copy to `/kaggle/working/rft_train.jsonl`
  at notebook start.

## Settings

- **Internet: OFF**
- Accelerator: GPU (RTX 6000 Pro Blackwell)

## What to do if you DO time out

The notebook has resume support: if cell 7 starts and finds existing rows
in `raw_generations_round2.jsonl`, it skips those `prompt_idx` and only
generates the remaining ones. Just re-run cell 7 (and cell 5 if the
kernel was restarted) in a fresh session.

In [ ]:
# ============================================================
# 1. SETUP — extract bundle that ships vLLM (offline)
# ============================================================
# The tinker-submission-notebook AND the official baseline-evaluation
# notebook both extract this exact bundle and successfully import vLLM
# from /tmp/vllm/. Bundle path uses UNDERSCORES not hyphens, and lives
# under notebooks/metric/.
import subprocess, sys, os, glob, importlib

# Prefer the bundle that the working notebooks use (it ships vLLM).
# Fall back to other locations if Kaggle reorganized things.
CANDIDATE_BUNDLES = [
    "/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script",  # ← has vLLM
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script",
    "/kaggle/usr/lib/nvidia-metric-utility-script",                   # stripped, no vLLM
]
bundle = next((b for b in CANDIDATE_BUNDLES if os.path.isdir(b)), None)
if bundle is None:
    raise FileNotFoundError(
        "No NVIDIA utility-script bundle found. Searched:\n  "
        + "\n  ".join(CANDIDATE_BUNDLES) +
        "\nEnable the competition metric so the bundle auto-attaches."
    )
print(f"[ok] Using bundle: {bundle}")

# Step 1: kill Kaggle's pre-installed torch (METH_CLASS crash with bundle's
# transformers 5.3 if both are loadable)
subprocess.run(
    "uv pip uninstall torch torchvision torchaudio || "
    "pip uninstall -y torch torchvision torchaudio || true",
    shell=True, check=False
)

# Step 2: extract bundle (flat layout — packages land directly in /tmp/)
print(f"Extracting bundle to /tmp ...")
r = subprocess.run(
    f"tar -cf - -C {bundle} . | tar -xf - -C /tmp",
    shell=True, capture_output=True, text=True
)
if r.returncode != 0:
    print(f"[warn] tar stderr: {r.stderr[:300]}")

# Step 3: ptxas binaries
for ptxas in ["/tmp/triton/backends/nvidia/bin/ptxas",
              "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"]:
    if os.path.exists(ptxas):
        subprocess.run(f"chmod +x {ptxas}", shell=True)

# Step 4: confirm vLLM is in /tmp
vllm_init = "/tmp/vllm/__init__.py"
if not os.path.exists(vllm_init):
    # search for it elsewhere
    found = glob.glob("/tmp/**/vllm/__init__.py", recursive=True) + \
            glob.glob("/kaggle/usr/lib/**/vllm/__init__.py", recursive=True)
    raise ImportError(
        f"vLLM not found at /tmp/vllm/.\n"
        f"Other vllm/ locations: {found}\n"
        f"You probably attached the wrong bundle. The one with vLLM is at:\n"
        f"  /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script\n"
        f"Enable the competition metric (auto-attaches it) and restart the kernel."
    )
print(f"[ok] vLLM present at /tmp/vllm/")

# Step 5: prune sys.path of competing locations and put /tmp FIRST
PRUNE_HINTS = ("ryanholbrook/nvidia_utility_script", "/kaggle/working/packages")
sys.path = [p for p in sys.path if not any(h in p for h in PRUNE_HINTS)]
sys.path = ["/tmp"] + [p for p in sys.path if p != "/tmp"]
print(f"sys.path[:3] = {sys.path[:3]}")

# Step 6: evict cached modules so /tmp versions win
for _m in list(sys.modules):
    top = _m.split(".")[0]
    if top in (
        "torch", "torchvision", "torchaudio", "torchgen", "functorch",
        "transformers", "tokenizers", "safetensors", "huggingface_hub",
        "accelerate", "peft", "datasets", "triton",
        "mamba_ssm", "causal_conv1d", "flash_attn", "vllm",
    ):
        del sys.modules[_m]

# Step 7: verify imports come from /tmp
print("\nVerifying imports:")
import torch
print(f"  torch        {torch.__version__:25s}  {torch.__file__}")
import transformers
print(f"  transformers {transformers.__version__:25s}  {transformers.__file__}")
import vllm
print(f"  vllm         {vllm.__version__:25s}  {vllm.__file__}")

assert torch.__file__.startswith("/tmp/"),       f"torch not from /tmp: {torch.__file__}"
assert transformers.__file__.startswith("/tmp/"), f"transformers not from /tmp: {transformers.__file__}"
assert vllm.__file__.startswith("/tmp/"),         f"vllm not from /tmp: {vllm.__file__}"

print(f"\n[ok] cell 1 done — vLLM {vllm.__version__}, torch {torch.__version__}, "
      f"CUDA: {torch.cuda.is_available()}")

In [2]:
# ============================================================
# 2. IMPORTS & ENVIRONMENT
# ============================================================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, time, re, hashlib
from collections import Counter, defaultdict
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from safetensors.torch import load_file as load_safetensors

print(f"PyTorch      : {torch.__version__}")
print(f"GPU          : {torch.cuda.get_device_name(0)}")
print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
import transformers
print(f"transformers : {transformers.__version__}")

# Mamba fast path check
try:
    import causal_conv1d, mamba_ssm
    print(f"causal_conv1d: {causal_conv1d.__version__}  ← Mamba fast path ON")
    print(f"mamba_ssm    : {mamba_ssm.__version__}")
except ImportError as e:
    print(f"[warn] Mamba fast path OFF — {e}")

PyTorch      : 2.12.0.dev20260324+cu128
GPU          : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM         : 102.0 GB
transformers : 5.3.0
[warn] Mamba fast path OFF — No module named 'cutlass'


In [3]:
# ============================================================
# 3. TRITON ENV — point at extracted ptxas (offline)
# ============================================================
# The utility script extraction in cell 1 already put ptxas-blackwell at
# /tmp/triton/backends/nvidia/bin/ptxas-blackwell. Just point env vars at it.
import os
PTXAS_PATH = "/tmp/triton/backends/nvidia/bin/ptxas-blackwell"
if os.path.exists(PTXAS_PATH):
    for v in ("TRITON_PTXAS_PATH", "TRITON_PTXAS_BLACKWELL_PATH",
              "TRITON_PTXAS_BIN", "TRITON_PTXAS"):
        os.environ[v] = PTXAS_PATH
    print(f"[ok] ptxas pointed at {PTXAS_PATH}")
else:
    # Fall back to non-Blackwell ptxas
    PTXAS_PATH = "/tmp/triton/backends/nvidia/bin/ptxas"
    if os.path.exists(PTXAS_PATH):
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_PATH
        print(f"[ok] ptxas (non-blackwell) pointed at {PTXAS_PATH}")
    else:
        print(f"[warn] no ptxas found in /tmp/triton")


[ok] ptxas pointed at /tmp/triton/backends/nvidia/bin/ptxas-blackwell


In [ ]:
# ============================================================
# 4. CONFIG — ROUND 2 TIGHT (designed to FIT in 12-hr session)
# ============================================================
# Round 1 results:
#   equation_numeric_deduce  : 26.2% accept ✓ (kept 567 — DONE, skip in round 2)
#   equation_numeric_guess   :  7.9% accept (kept 35  — could use ~50 more)
#   bit_manipulation         :  1.1% accept (kept 60  — TRUNCATED at 6144 tokens)
#   cryptarithm_deduce       :  1.2% accept (kept 32  — TRUNCATED at 6144 tokens)
#   cryptarithm_guess        :  1.4% accept (kept  9  — TRUNCATED at 6144 tokens)
#
# First round 2 attempt (K=6, max_tok=14336, all 2298 prompts) ran past 12 hrs.
# This config is sized to FINISH in a single 12-hr session:
#   - K=4 (was 6)                     — 33% fewer completions
#   - MAX_NEW_TOK=10240 (was 14336)   — 28% smaller worst-case output
#   - MAX_PROMPTS_PER_CATEGORY=400    — caps bit_manip at 400 of 1364
#   - max_model_len=12288             — smaller KV cache, fits more concurrent
#   - CHUNK_SIZE=200 with periodic save — if you DO time out, progress is kept
#
# Worst-case math: ~1075 prompts × K=4 = 4300 completions × ~6K avg tokens
# ≈ 26M tokens, expected ~5-8 hrs. Safe within 12 hr limit.

MODEL_PATH    = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"

# *** EDIT THIS *** — path to YOUR 0.86 adapter dataset
ADAPTER_PATH  = "/kaggle/input/models/manish756/nvidia-adapter/transformers/default/7"

DATA_DIR_CANDIDATES = [
    "/kaggle/input/datasets/manish756/nemotron-dataset/all_categorical_splits",
]

# Round 2: skip equation_numeric_deduce (already 567 ≫ enough). Focus on the 4
# categories that need more correct samples.
HARD_CATEGORIES = [
    "train_cot_bit_manipulation.jsonl",       # capped at 400 (was 1364)
    "train_cot_cryptarithm_deduce.jsonl",     # capped at 400 (of 659)
    "train_cot_cryptarithm_guess.jsonl",      # 164 (under cap, all used)
    "train_cot_equation_numeric_guess.jsonl", # 111 (under cap, all used)
]

# ============================================================
# Generation settings — TIGHT 12-hr budget
# ============================================================
K            = 4         # was 6 — reduced 33% to fit budget
TEMPERATURE  = 0.5       # focused reasoning (proven in round 1 diagnostics)
TOP_P        = 0.9       # stricter nucleus
MAX_NEW_TOK  = 10240     # was 14336 — still 67% more than round 1's 6144

# CRITICAL: cap prompts per category so total work fits in 12-hr session.
# bit_manipulation has 1364 unique prompts; this caps it at 400.
# Other categories are smaller, so cap doesn't reduce them.
MAX_PROMPTS_PER_CATEGORY = 400

# ============================================================
# Output paths — keep round 1 file untouched, write round 2 separately
# ============================================================
RAW_OUTPUT      = "/kaggle/working/raw_generations_round2.jsonl"
RFT_TRAIN_FILE  = "/kaggle/working/rft_train_round2.jsonl"

# Generation chunk size for periodic checkpointing (cell 7 uses this).
# Smaller = more frequent saves = better timeout safety, slightly more overhead.
CHUNK_SIZE = 200

print(f"Adapter        : {ADAPTER_PATH}")
print(f"Hard cats (4)  : {HARD_CATEGORIES}")
print(f"K              : {K}    temp={TEMPERATURE}  top_p={TOP_P}  max_new={MAX_NEW_TOK}")
print(f"Per-cat cap    : {MAX_PROMPTS_PER_CATEGORY}  (bit_manip 1364 → {MAX_PROMPTS_PER_CATEGORY})")
print(f"Chunk size     : {CHUNK_SIZE}  (saves progress every {CHUNK_SIZE} prompts)")
print(f"Output         : {RAW_OUTPUT}")

In [ ]:
# ============================================================
# 5. LOAD vLLM ENGINE + LoRA ADAPTER
# ============================================================
# vLLM with continuous batching + LoRARequest. Same pattern as the
# tinker-submission and baseline-evaluation notebooks.
#
# ROUND 2 TIGHT: max_model_len=12288 (was 16384) saves KV cache memory
# so we can fit more concurrent sequences (max_num_seqs=48).
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from transformers import AutoTokenizer

# Verify adapter
adapter_cfg = os.path.join(ADAPTER_PATH, "adapter_config.json")
adapter_w   = os.path.join(ADAPTER_PATH, "adapter_model.safetensors")
assert os.path.exists(adapter_cfg) and os.path.exists(adapter_w), (
    f"adapter files not found at {ADAPTER_PATH}\nEdit ADAPTER_PATH in cell 4."
)
with open(adapter_cfg) as f:
    cfg = json.load(f)
print(f"Adapter:  r={cfg.get('r')}  alpha={cfg.get('lora_alpha')}  "
      f"target={cfg.get('target_modules')}")
LORA_RANK_FROM_CFG = cfg.get("r", 32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

print("\nLoading vLLM engine (this takes ~3-4 min) ...")
llm = LLM(
    model                  = MODEL_PATH,
    trust_remote_code      = True,
    dtype                  = "bfloat16",
    enable_lora            = True,
    max_lora_rank          = max(LORA_RANK_FROM_CFG, 32),
    max_loras              = 1,
    max_num_seqs           = 48,         # smaller max_model_len → more concurrent
    gpu_memory_utilization = 0.85,
    max_model_len          = 12288,      # covers ~2K prompt + 10240 output
    enable_prefix_caching  = True,
    enable_chunked_prefill = True,
)

lora_req = LoRARequest(
    lora_name   = "rft_adapter",
    lora_int_id = 1,
    lora_path   = ADAPTER_PATH,
)

print("\nSanity test (1 prompt, deterministic, 32 tokens):")
sp_test = SamplingParams(n=1, temperature=0.0, max_tokens=32)
test_outputs = llm.generate(
    ["Say hello in three words."],
    sampling_params = sp_test,
    lora_request    = lora_req,
)
print(f"  output: {test_outputs[0].outputs[0].text!r}")
print("[ok] vLLM engine + LoRA adapter ready")

In [6]:
# ============================================================
# 6. LOAD PROMPTS (hard categories only) + ground-truth answers
# ============================================================
data_dir = None
for c in DATA_DIR_CANDIDATES:
    if c and os.path.isdir(c) and any(
        os.path.exists(os.path.join(c, f)) for f in HARD_CATEGORIES
    ):
        data_dir = c; break
assert data_dir, f"no data dir found; searched: {DATA_DIR_CANDIDATES}"
print(f"Data dir: {data_dir}")

def extract_boxed(text):
    """Return the LAST \\boxed{...} content. None if missing."""
    matches = re.findall(r"\\boxed\{([^}]*)\}", text)
    return matches[-1].strip() if matches else None

prompts = []
for fname in HARD_CATEGORIES:
    fpath = os.path.join(data_dir, fname)
    if not os.path.exists(fpath):
        print(f"  [skip] {fname}"); continue
    cat = fname.replace("train_cot_", "").replace(".jsonl", "")
    n_loaded = 0; n_skipped = 0
    with open(fpath) as f:
        for line in f:
            if not line.strip(): continue
            r = json.loads(line)
            msgs = [m for m in r["messages"] if m["role"] != "system"]
            if not msgs or msgs[-1]["role"] != "assistant":
                n_skipped += 1; continue
            user = msgs[0]["content"]
            asst_orig = msgs[-1]["content"]
            gt = extract_boxed(asst_orig)
            if gt is None:
                n_skipped += 1; continue
            prompts.append({
                "category": cat,
                "user": user,
                "gt_answer": gt,
            })
            n_loaded += 1
            if MAX_PROMPTS_PER_CATEGORY and n_loaded >= MAX_PROMPTS_PER_CATEGORY:
                break
    print(f"  {n_loaded:>5} from {fname}  (skipped {n_skipped})")

# Dedupe by user prompt — RFT only needs unique prompts
seen = set(); unique = []
for p in prompts:
    key = hashlib.md5(p["user"].encode()).hexdigest()
    if key in seen: continue
    seen.add(key); unique.append(p)
print(f"\nTotal: {len(prompts)} → {len(unique)} unique prompts after dedup")
prompts = unique

# Per-category breakdown
cat_counts = Counter(p["category"] for p in prompts)
print("\nPer-category prompt counts:")
for c, n in cat_counts.most_common():
    print(f"  {c:30s} {n:>5}")

# Estimated generation time
SEC_PER_GEN = 5  # rough estimate at seq up to 4k tokens on RTX 6000 Pro
est_hrs = len(prompts) * K * SEC_PER_GEN / 3600
print(f"\nEstimated runtime: {len(prompts)} prompts × K={K} × ~{SEC_PER_GEN}s ≈ {est_hrs:.1f} hrs")


Data dir: /kaggle/input/datasets/manish756/nemotron-dataset/all_categorical_splits
   2728 from train_cot_bit_manipulation.jsonl  (skipped 0)
    540 from train_cot_equation_numeric_deduce.jsonl  (skipped 0)
    659 from train_cot_cryptarithm_deduce.jsonl  (skipped 0)
    164 from train_cot_cryptarithm_guess.jsonl  (skipped 0)
    111 from train_cot_equation_numeric_guess.jsonl  (skipped 0)

Total: 4202 → 2838 unique prompts after dedup

Per-category prompt counts:
  bit_manipulation                1364
  cryptarithm_deduce               659
  equation_numeric_deduce          540
  cryptarithm_guess                164
  equation_numeric_guess           111

Estimated runtime: 2838 prompts × K=4 × ~5s ≈ 15.8 hrs


In [ ]:
# ============================================================
# 7. GENERATE — CHUNKED vLLM batched inference (timeout-safe)
# ============================================================
# CRITICAL change vs round 1: process prompts in chunks of CHUNK_SIZE
# (set in cell 4). After each chunk we WRITE TO DISK before moving on.
#
# Why: a single big llm.generate() call holds all 4300 outputs in memory
# until completion. If the kernel hits the 12-hr limit mid-call, ZERO
# results are saved. With chunked processing, every CHUNK_SIZE prompts
# are flushed to /kaggle/working/raw_generations_round2.jsonl — so a
# timeout only loses the in-flight chunk, not everything.
#
# vLLM still does continuous batching INSIDE each chunk (via SamplingParams
# with n=K), so per-chunk speed is the same as one big call.

# ---- Resume support: skip prompts already in RAW_OUTPUT ----
done_idx = set()
if os.path.exists(RAW_OUTPUT):
    with open(RAW_OUTPUT) as f:
        for line in f:
            try: done_idx.add(json.loads(line)["prompt_idx"])
            except: pass
    print(f"Resuming: {len(done_idx)} done, {len(prompts)-len(done_idx)} remaining")
else:
    print(f"Fresh run: {len(prompts)} prompts")

# Build templated prompt strings, remember original indices
def make_prompt(user_text):
    msgs = [{"role": "user", "content": user_text}]
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True,
            enable_thinking=True,   # Nemotron-H reasoning mode
        )
    except TypeError:
        try:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            return (f"<|im_start|>user\n{user_text}<|im_end|>\n"
                    f"<|im_start|>assistant\n")

todo = [(i, p) for i, p in enumerate(prompts) if i not in done_idx]

if not todo:
    print("Nothing to generate. Skip to cell 8.")
else:
    print(f"\nGenerating {len(todo)} prompts × K={K} samples each ...")
    print(f"Sampling : temp={TEMPERATURE}  top_p={TOP_P}  max_tokens={MAX_NEW_TOK}")
    print(f"Chunking : {CHUNK_SIZE} prompts/chunk  → {-(-len(todo)//CHUNK_SIZE)} chunks total\n")

    # Stop tokens — vLLM will end generation when it sees any of these
    stop_strs = ["<|im_end|>", "<|endoftext|>"]
    if tokenizer.eos_token and tokenizer.eos_token not in stop_strs:
        stop_strs.append(tokenizer.eos_token)

    sp = SamplingParams(
        n           = K,
        temperature = TEMPERATURE,
        top_p       = TOP_P,
        max_tokens  = MAX_NEW_TOK,
        stop        = stop_strs,
    )

    t_start = time.time()
    n_total_written = 0

    # Open in append mode so resume + chunk writes accumulate
    with open(RAW_OUTPUT, "a") as out_file:
        for chunk_start in range(0, len(todo), CHUNK_SIZE):
            chunk = todo[chunk_start : chunk_start + CHUNK_SIZE]
            chunk_prompts = [make_prompt(p["user"]) for _, p in chunk]

            t_chunk = time.time()
            chunk_outputs = llm.generate(
                chunk_prompts,
                sampling_params = sp,
                lora_request    = lora_req,
            )

            # Write this chunk to disk IMMEDIATELY
            for (orig_idx, p), vllm_out in zip(chunk, chunk_outputs):
                completions = [o.text for o in vllm_out.outputs]
                rec = {
                    "prompt_idx":  orig_idx,
                    "category":    p["category"],
                    "user":        p["user"],
                    "gt_answer":   p["gt_answer"],
                    "completions": completions,
                }
                out_file.write(json.dumps(rec, ensure_ascii=False) + "\n")
                n_total_written += 1
            out_file.flush()
            os.fsync(out_file.fileno())   # ensure on-disk persistence

            # Progress
            chunk_time = time.time() - t_chunk
            done = chunk_start + len(chunk)
            elapsed_min = (time.time() - t_start) / 60
            rate = done / max(time.time() - t_start, 1)
            eta_min = (len(todo) - done) / max(rate, 1e-9) / 60
            print(f"  [chunk {chunk_start//CHUNK_SIZE + 1}/{-(-len(todo)//CHUNK_SIZE)}]  "
                  f"{done}/{len(todo)} done  "
                  f"({chunk_time/60:.1f} min/chunk, "
                  f"elapsed {elapsed_min:.0f} min, ETA {eta_min:.0f} min)  "
                  f"→ {n_total_written} records on disk")

    elapsed = time.time() - t_start
    print(f"\n[ok] vLLM generation done in {elapsed/60:.1f} min "
          f"({n_total_written * K / max(elapsed, 1):.1f} completions/sec)")
    print(f"Wrote {n_total_written} new records to {RAW_OUTPUT}")

In [ ]:
# ============================================================
# 8. FILTER — extract \boxed{}, compare to GT, dedupe, save SFT JSONL
# ============================================================
def normalize_answer(s):
    """Loose match — strip whitespace, case, surrounding quotes."""
    if s is None: return ""
    s = str(s).strip().lower()
    s = s.strip("\"' ")
    s = s.replace(" ", "")
    return s

stats = Counter()
seen_pairs = set()
keep = []

with open(RAW_OUTPUT) as f:
    for line in f:
        try:
            r = json.loads(line)
        except Exception:
            stats["bad_json"] += 1; continue

        gt = normalize_answer(r["gt_answer"])
        cat = r["category"]
        stats[f"prompt_{cat}"] += 1

        for comp in r["completions"]:
            stats["total_completions"] += 1
            pred = extract_boxed(comp)
            if pred is None:
                stats["no_box"] += 1; continue
            if normalize_answer(pred) != gt:
                stats["wrong"] += 1; continue

            # Dedup: same (prompt, completion) pair shouldn't appear twice
            h = hashlib.md5((str(r["prompt_idx"]) + comp).encode()).hexdigest()
            if h in seen_pairs:
                stats["dup_completion"] += 1; continue
            seen_pairs.add(h)

            stats["kept"] += 1
            stats[f"kept_{cat}"] += 1

            keep.append({
                "messages": [
                    {"role": "user",      "content": r["user"]},
                    {"role": "assistant", "content": comp.strip()},
                ],
                "category": cat,
            })

with open(RFT_TRAIN_FILE, "w") as f:
    for r in keep:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Wrote {len(keep)} training examples to {RFT_TRAIN_FILE}\n")
print(f"{'='*60}")
print(f"Stats")
print(f"{'='*60}")
for k in ["total_completions", "no_box", "wrong", "dup_completion", "kept"]:
    if k in stats: print(f"  {k:25s} {stats[k]:>6}")

# Acceptance rate
if stats["total_completions"] > 0:
    rate = 100 * stats["kept"] / stats["total_completions"]
    print(f"\n  Acceptance rate         : {rate:.1f}%  (kept / total)")

print(f"\n{'='*60}")
print(f"Per-category yield")
print(f"{'='*60}")
print(f"  {'category':30s} {'prompts':>8} {'kept':>6} {'avg':>5}")
for cat in sorted(set(k.replace("prompt_","") for k in stats if k.startswith("prompt_"))):
    np = stats[f"prompt_{cat}"]
    nk = stats[f"kept_{cat}"]
    avg = nk / np if np else 0
    print(f"  {cat:30s} {np:>8} {nk:>6} {avg:>5.2f}")


In [ ]:
# ============================================================
# 9. COMBINE — merge round 1 + round 2 RFT into single training file
# ============================================================
# This produces /kaggle/working/rft_train_combined.jsonl which goes into
# your v76 training notebook. Run this AFTER cell 8 has written
# rft_train_round2.jsonl.

import os, json, hashlib

# Round 1 output (from previous run, must be uploaded as Kaggle dataset
# OR copied to /kaggle/working/ at notebook start)
ROUND1_PATHS = [
    "/kaggle/working/rft_train.jsonl",                            # if copied here
    "/kaggle/input/manish-rft-train-round1/rft_train.jsonl",      # or attached as dataset
    "/kaggle/input/datasets/manish756/rft-train-round1/rft_train.jsonl",
]
ROUND2_PATH = RFT_TRAIN_FILE  # from cell 4 = rft_train_round2.jsonl
COMBINED    = "/kaggle/working/rft_train_combined.jsonl"

round1_path = next((p for p in ROUND1_PATHS if os.path.exists(p)), None)

combined = []
seen_hashes = set()  # dedup across rounds

def add_records(path, label):
    if not path or not os.path.exists(path):
        print(f"  [skip] {label} — not found at {path}")
        return 0
    n_new = 0; n_dup = 0
    with open(path) as f:
        for line in f:
            try:
                r = json.loads(line)
            except Exception:
                continue
            user = r["messages"][0]["content"] if r.get("messages") else ""
            asst = r["messages"][-1]["content"] if r.get("messages") else ""
            h = hashlib.md5((user + asst).encode()).hexdigest()
            if h in seen_hashes:
                n_dup += 1; continue
            seen_hashes.add(h)
            combined.append(r)
            n_new += 1
    print(f"  [{label}] {path}: +{n_new} new, {n_dup} dups dropped")
    return n_new

print("Combining RFT rounds:")
add_records(round1_path, "round 1")
add_records(ROUND2_PATH, "round 2")

# Per-category counts in the final combined set
from collections import Counter
cat_counts = Counter(r.get("category", "?") for r in combined)
print(f"\nCombined: {len(combined)} unique training examples")
print(f"\nPer-category breakdown:")
for cat, n in cat_counts.most_common():
    print(f"  {cat:30s} {n:>5}")

with open(COMBINED, "w") as f:
    for r in combined:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"\n[ok] Wrote {COMBINED}")
print(f"\n{'='*60}")
print("NEXT STEPS")
print(f"{'='*60}")
print(f"""
1. Download {COMBINED}

2. Upload as a new Kaggle dataset (e.g. 'manish-nemotron-rft-combined')

3. In your v76 training notebook (cell 5 hyperparameters):
   - Add the new dataset to DATA_DIR_CANDIDATES
   - Update CATEGORY_FILES to use 'rft_train_combined.jsonl' for hard cats
     (keep cipher/unit_conversion/numeral/gravity files for easy cats)
   - Settings:
       LORA_ALPHA   = 48     (proven from 0.86 run)
       LR           = 5e-5   (proven)
       NUM_EPOCHS   = 1      (RFT data is cleaner; don't overfit)
       WARMUP_STEPS = 50
       MAX_SEQ_LEN  = 12288  (cover long RFT CoTs without tail-truncation)

4. Train one fresh epoch (~3-5 hrs)

5. Submit. Expected LB: 0.87-0.89 (vs 0.86 baseline)
""")
